# 01 · Setup & Cluster Basics

> **Module 2 · Spark Fundamentals · Lecture 2.1**

In this notebook you will:

- Install and verify PySpark, then build a `SparkSession` with `SparkSession.builder`
- Understand the master modes `local[*]` and `local-cluster`, and read live cluster metadata
- Set the config knobs that matter most (`spark.sql.shuffle.partitions`) and find the Spark UI
- Read a physical plan with `df.explain()` and inspect a first DataFrame with `show()`, `printSchema()`, `describe()`
- Run the same notebook logic on Dataproc with `gcloud dataproc jobs submit pyspark` and `--py-files`

---


## Setup / prerequisites

- **Python 3.9–3.11** and **Java 11 or 17** on your machine.
- **PySpark 3.5.x**: `pip install "pyspark==3.5.1" pandas pyarrow numpy`
- This notebook is **self-contained and offline**: every code cell runs top-to-bottom with
  no GCP account. All data operations use local paths under `/tmp/`.
- GCS / BigQuery / Pub/Sub / JDBC code appears in **clearly-marked optional cells** that
  you uncomment after authenticating with `gcloud auth application-default login`.
- On Dataproc, **do not `pip install pyspark`** — use the Spark runtime already on the
  image (Spark 3.5 on Dataproc 2.2).


## 1 · Install and verify PySpark

Install locally for development; on Dataproc use the image's own Spark runtime.

```bash
pip install "pyspark==3.5.1" pandas pyarrow numpy
```

Verify the version matches the cluster (Spark 3.5.x on Dataproc 2.2).

In [ ]:
# pip install "pyspark==3.5.1" pandas pyarrow numpy
import pyspark
import sys
print("Python  :", sys.version.split()[0])
print("PySpark :", pyspark.__version__)

## 2 · Build a SparkSession

`SparkSession.builder` is the single entry point for DataFrames, SQL, and the catalog.
`.getOrCreate()` returns an existing session if one is already active — which is why the
same code is safe in a notebook and in a `spark-submit` script.

| Builder method | Purpose |
|---|---|
| `.master("local[*]")` | Run in this JVM with N threads. **Local development only.** |
| `.master("local-cluster[2,2,1024]")` | Simulated multi-node cluster: 2 workers, 2 cores, 1 GB each. |
| `.appName("...")` | Labels the application in the Spark UI and Dataproc UI. |
| `.config(k, v)` | Sets a Spark configuration. |
| `.getOrCreate()` | Returns the active session or creates one. |

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pyspark

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("notebook-01")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("PySpark :", pyspark.__version__)
print("Spark   :", spark.version)
print("Master  :", spark.sparkContext.master)


### Master modes: `local[*]` vs `local-cluster`

- `local[*]` — one JVM, N task threads, everything in one process. Fast to start, no
  network, no real distribution. This is what the course uses by default.
- `local-cluster[N, C, M]` — N worker JVMs, C cores each, M MB of memory each. It
  exercises the *real* driver/executor protocol (RPC, block manager, task serialisation)
  on a laptop, so a job that only works in `local[*]` fails here first.
- **On Dataproc omit `master` entirely.** Spark then uses the cluster manager (`yarn`).
  Hardcoding `master="local[*]"` in a submitted script makes the whole job run on the
  driver node and ignores every worker you paid for.

In [ ]:
# Compare the two local master modes.
from pyspark.sql import SparkSession

spark_local_cluster = (
    SparkSession.builder
    .master("local-cluster[2,2,1024]")
    .appName("notebook-01-local-cluster")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
print("local-cluster master :", spark_local_cluster.sparkContext.master)
print("parallelism         :", spark_local_cluster.sparkContext.defaultParallelism)
print("UI URL              :", spark_local_cluster.sparkContext.uiWebUrl)
spark_local_cluster.stop()

# The original local[*] session is still the active one for the rest of the notebook.
spark = SparkSession.builder.getOrCreate()
print("restored master     :", spark.sparkContext.master)

## 3 · Read live cluster metadata

Never guess cluster shape — ask the running application.

In [ ]:
sc = spark.sparkContext

print("Spark version      :", spark.version)
print("App name           :", sc.appName)
print("Master             :", sc.master)
print("Default parallelism:", sc.defaultParallelism)
print("Application ID     :", sc.applicationId)
print("Executor memory    :", sc.getConf().get("spark.executor.memory", "n/a"))
print("Executor cores     :", sc.getConf().get("spark.executor.cores", "n/a"))
print("Shuffle partitions :", spark.conf.get("spark.sql.shuffle.partitions"))
print("Session timezone   :", spark.conf.get("spark.sql.session.timeZone"))

# Everything you set on the builder or passed with --conf is visible here.
print("\nSelected SparkConf entries:")
for key in ["spark.sql.shuffle.partitions", "spark.sql.adaptive.enabled",
            "spark.sql.execution.arrow.pyspark.enabled", "spark.serializer"]:
    try:
        print(f"  {key:50s} = {spark.conf.get(key)}")
    except Exception as exc:                      # pragma: no cover - defensive
        print(f"  {key:50s} = <unavailable: {exc}>")

## 4 · Config knobs

| Config | Default | Why it matters |
|---|---|---|
| `spark.sql.shuffle.partitions` | `200` | Reduce-side task count for **every** shuffle. 10 rows still create 200 tasks. |
| `spark.sql.adaptive.enabled` | `true` | AQE — coalesces small shuffle partitions and can convert joins to broadcast at runtime. |
| `spark.sql.autoBroadcastJoinThreshold` | `10485760` (10 MB) | Above this, the small side of a join is shuffled rather than broadcast. |
| `spark.sql.session.timeZone` | session/local | A correctness setting, not a formatting one. Pin it to `UTC`. |
| `spark.sql.execution.arrow.pyspark.enabled` | `false` | Required for `pandas_udf` and makes `toPandas()` much faster. |
| `spark.executor.memory` | `1g` (local) | Executor JVM heap. **Cannot be changed at runtime** — set it at submit time. |
| `spark.driver.memory` | `1g` | Driver JVM heap. **Cannot be changed at runtime.** |
| `spark.memory.fraction` | `0.6` | Share of executor heap reserved for execution vs. storage. |

In [ ]:
# Live, mutable configuration.
spark.conf.set("spark.sql.shuffle.partitions", "8")
print("shuffle partitions now:", spark.conf.get("spark.sql.shuffle.partitions"))

# Static configuration can be inspected but not changed after startup.
try:
    spark.conf.set("spark.executor.memory", "4g")
except Exception as exc:
    print("executor.memory is static ->", type(exc).__name__)
    print(str(exc).splitlines()[0])

# Reset for the rest of the notebook.
spark.conf.set("spark.sql.shuffle.partitions", "4")

## 5 · Spark UI

The Spark UI shows the jobs, stages, tasks, and SQL plans that the driver produced.
Locate it with `spark.sparkContext.uiWebUrl`:

- **Local mode** — normally `http://localhost:4040` (the port increments if 4040 is taken).
- **Dataproc** — the Component Gateway proxies the UI; open the cluster's
  **Spark UI / Jupyter** link, or the `Spark UI` tab in the Dataproc console.

In the UI, the **Stages** tab shows **Input Size / Records** and **Shuffle Read/Write** —
the fastest way to see whether a filter was pushed down and how much data crossed the network.

In [ ]:
ui_url = spark.sparkContext.uiWebUrl
print("Spark UI:", ui_url)
print()
print("Open that URL while this session is alive to watch jobs and stages appear.")
print("Tabs worth checking:")
print("  Jobs    -> one row per action (show/count/write)")
print("  Stages  -> task counts, shuffle bytes, input bytes")
print("  SQL     -> the physical plan for each DataFrame action")
print("  Storage -> cached/persisted blocks")

## 6 · Your first DataFrame

`spark.createDataFrame` turns a local Python list into a distributed DataFrame.
Transformations are **lazy**; only actions (`show`, `count`, `write`) execute a job.

In [ ]:
# ------------------------------------------------------------------
# Self-contained sample data: a tiny POS-style dataset.
# ------------------------------------------------------------------
sample_rows = [
    ("T-001", "C-101", "Widget",   2,  19.99, "2026-10-01"),
    ("T-001", "C-102", "Widget",   1,  19.99, "2026-10-01"),
    ("T-002", "C-101", "Gadget",   3,  49.50, "2026-10-01"),
    ("T-003", "C-103", "Gizmo",    1, 129.00, "2026-10-02"),
    ("T-004", "C-102", "Widget",   5,  19.99, "2026-10-02"),
    ("T-005", "C-104", "Gadget",   2,  49.50, "2026-10-02"),
]
sample_df = spark.createDataFrame(
    sample_rows,
    schema="transaction_id string, customer_id string, product string, "
           "quantity int, unit_price double, sale_date string",
)
sample_df.printSchema()
sample_df.show(truncate=False)


### Laziness, demonstrated

Building the plan costs nothing. The job only appears in the Spark UI after an action.

In [ ]:
filtered = sample_df.filter(F.col("quantity") >= 2)   # lazy: plan only
print("Logical plan preview:")
print(filtered._jdf.queryExecution().logical().toString()[:300])
print()
print("Now run an action ->")
filtered.show(truncate=False)

## 7 · `df.explain()`

`explain()` is free — it builds the optimized physical plan without running anything.
Read it bottom-up and count the `Exchange` nodes: each one is a full network shuffle.

| Call | Output |
|---|---|
| `df.explain()` | Simple tree (equivalent to `mode="simple"`). |
| `df.explain(mode="formatted")` | Tree with `PushedFilters` and `ReadSchema`. |
| `df.explain(True)` | Identical to `mode="formatted"`. |

In [ ]:
sample_df.filter(F.col("quantity") >= 2).explain()

In [ ]:
sample_df.filter(F.col("quantity") >= 2).explain(mode="formatted")

## 8 · Inspecting data

- `show()` — formatted preview (an action).
- `printSchema()` — the schema tree.
- `describe()` — summary statistics for numeric columns.
- `dtypes` — `(name, type)` pairs.
- `columns` — the column names.
- `count()` — row count, computed without shipping rows to the driver.

In [ ]:
print("== show() ==")
sample_df.show(3, truncate=False)

print("\n== printSchema() ==")
sample_df.printSchema()

print("\n== describe() ==")
sample_df.describe().show(truncate=False)

print("\n== dtypes / columns / count ==")
print(sample_df.dtypes)
print(sample_df.columns)
print("rows:", sample_df.count())

## Running on Dataproc vs local

The **same Python code** runs in both places. Only the session construction differs.

| | Local | Dataproc cluster |
|---|---|---|
| Master | `local[*]` (set in code) | `yarn` (omit `master`) |
| Concurrency | One JVM, N threads | Many JVMs across many nodes |
| Realistic volume | Hundreds of MB | Terabytes |
| `parallelize` of a list | Cheap (in-process) | Shipped over the network |
| Failure handling | None | Task retry, executor replacement |
| Python modules | Local filesystem | **Must be shipped with `--py-files`** |

### Submitting a PySpark job to Dataproc

```bash
# Upload your script (and any shared modules) to GCS first.
gcloud storage cp jobs/etl_main.py      gs://YOUR_BUCKET/jobs/
gcloud storage cp jobs/common.py         gs://YOUR_BUCKET/jobs/

# Submit. The script must NOT hardcode master="local[*]".
gcloud dataproc jobs submit pyspark gs://YOUR_BUCKET/jobs/etl_main.py \
  --project=YOUR_PROJECT_ID \
  --region=us-central1 \
  --cluster=your-dataproc-cluster \
  --py-files=gs://YOUR_BUCKET/jobs/common.py \
  --jars=gs://YOUR_BUCKET/libs/spark-avro_2.12-3.5.3.jar \
  --driver-memory=4g \
  --executor-memory-overhead=2g \
  --properties=spark.sql.shuffle.partitions=200,spark.sql.adaptive.enabled=true
```

**`--py-files` is not optional.** Workers do not have your laptop's filesystem. Any
module you `import` (or any `.zip`/`.egg` dependency) must be shipped with `--py-files`.
`--jars` does the same for Java libraries such as the Avro or BigQuery connectors.

### Notebook on Dataproc

Dataproc images with the `JUPYTER` component give you a Jupyter environment on the
master. Open it through the **Component Gateway**; the notebook then runs *on the cluster*
against the real `yarn` master.

In [ ]:
# The portable session pattern used everywhere in this course.
import sys

def build_session(app_name):
    builder = SparkSession.builder.appName(app_name)
    if "spark-submit" not in sys.argv[0] and "notebook" not in sys.argv[0]:
        builder = builder.master("local[*]")
    return builder.getOrCreate()

# On Dataproc you would call build_session("etl") with no local master.
print("Detected master:", spark.sparkContext.master)
print("Detected parallelism:", spark.sparkContext.defaultParallelism)

## 🔑 Key Takeaways

- Verify the PySpark version first — the pip package must match the cluster's Spark version.
- `SparkSession.builder.getOrCreate()` is the only entry point you need; it is safe in notebooks and in submitted scripts.
- **Never hardcode `master` in a script you will submit to Dataproc.** `local[*]` makes the whole job run on the driver node.
- `local-cluster[N,C,M]` exercises the real driver/executor protocol on a laptop — use it to catch serialization bugs early.
- Read cluster metadata from `spark.sparkContext` (`appName`, `master`, `defaultParallelism`, `applicationId`) and from `spark.conf.get(...)`.
- `spark.sql.shuffle.partitions` (default 200) is the single most consequential knob; set it down for small data.
- `spark.executor.memory` and `spark.driver.memory` are **static** — they must be set at submit time, not with `spark.conf.set`.
- `df.explain(mode="formatted")` is free; count the `Exchange` nodes to measure shuffle complexity.
- Transformations are lazy; `show()`, `count()`, `write()`, and `collect()` are actions.
- On Dataproc, ship Python modules with `--py-files` and Java libraries with `--jars`.

---

## 📚 Next steps

- Run `01-setup-and-cluster-basics.ipynb` on a Dataproc Jupyter session and compare `sc.master` and `sc.defaultParallelism`.
- Re-run the `explain()` cells after changing `spark.sql.shuffle.partitions` and watch the number in the `Exchange` node change.
- Submit a minimal script with `gcloud dataproc jobs submit pyspark` and confirm the Spark UI shows a `yarn` master.
- Continue to `02-rdd-fundamentals.ipynb` for the RDD API and the DataFrame comparison table.
